# 01 — Exploração do Dataset

Objetivo: entender os dados antes de treinar qualquer modelo.

- Distribuição de classes
- Comprimento dos textos
- Como CodeBERT tokeniza um finding real
- Exemplos representativos por categoria

In [ ]:
import sys, json
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from transformers import AutoTokenizer

sys.path.insert(0, str(Path('../src').resolve()))
from model import MODEL_NAME, LABELS

SPLITS_DIR = Path('../data/splits')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

## 1. Carregar splits

In [ ]:
def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f]

train = load_jsonl(SPLITS_DIR / 'train.jsonl')
val   = load_jsonl(SPLITS_DIR / 'val.jsonl')
test  = load_jsonl(SPLITS_DIR / 'test.jsonl')

print(f'Train: {len(train)} | Val: {len(val)} | Test: {len(test)}')

## 2. Distribuição de classes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

for ax, split, name in zip(axes, [train, val, test], ['Train', 'Val', 'Test']):
    counts = Counter(e['label'] for e in split)
    ax.bar(counts.keys(), counts.values(), color=sns.color_palette('husl', len(LABELS)))
    ax.set_title(f'{name} ({sum(counts.values())} exemplos)')
    ax.set_xlabel('Label')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.suptitle('Distribuição de Classes por Split', y=1.02, fontsize=13)
plt.show()

## 3. Comprimento dos textos (em caracteres)

In [ ]:
df = pd.DataFrame(train)
df['text_len'] = df['text'].apply(len)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df['text_len'].hist(bins=40, ax=axes[0], color='steelblue')
axes[0].set_title('Distribuição de comprimento (chars)')
axes[0].set_xlabel('Chars')

df.groupby('label')['text_len'].mean().sort_values().plot.barh(ax=axes[1], color='coral')
axes[1].set_title('Comprimento médio por label')
axes[1].set_xlabel('Chars')

plt.tight_layout()
plt.show()

print(df['text_len'].describe())

## 4. Tokenization com CodeBERT

Esta célula responde: **como o modelo vê um finding?**

O tokenizer converte texto em IDs de tokens. O modelo nunca vê palavras — só números.
Tokens especiais `[CLS]` e `[SEP]` são adicionados automaticamente.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

example = "SQL query built with string concatenation — use parameterized queries to prevent injection"
tokens = tokenizer.tokenize(example)
encoding = tokenizer(example, return_tensors='pt')

print(f'Texto original ({len(example)} chars):')
print(f'  {example!r}')
print(f'\nTokens ({len(tokens)} tokens):')
print(f'  {tokens}')
print(f'\ninput_ids shape: {encoding["input_ids"].shape}')
print(f'  [CLS]={tokenizer.cls_token_id}, [SEP]={tokenizer.sep_token_id}')

## 5. Comprimento em tokens (o que realmente importa para o modelo)

In [ ]:
sample = train[:200]
token_lengths = []
for ex in sample:
    ids = tokenizer(ex['text'], truncation=False)['input_ids']
    token_lengths.append(len(ids))

import numpy as np
print(f'Comprimento em tokens (sample de {len(sample)}):')
print(f'  min={min(token_lengths)}, max={max(token_lengths)}, mean={np.mean(token_lengths):.1f}, p95={np.percentile(token_lengths, 95):.0f}')
print(f'\nExemplos > 256 tokens (nosso max_length): {sum(l > 256 for l in token_lengths)}')

plt.figure(figsize=(8, 4))
plt.hist(token_lengths, bins=30, color='mediumseagreen')
plt.axvline(256, color='red', linestyle='--', label='max_length=256')
plt.title('Distribuição de comprimento em tokens')
plt.xlabel('Tokens')
plt.legend()
plt.show()

## 6. Exemplos representativos por categoria

In [ ]:
from collections import defaultdict
import random

random.seed(42)
by_label = defaultdict(list)
for ex in train:
    by_label[ex['label']].append(ex['text'])

for label in LABELS:
    samples = random.sample(by_label[label], min(3, len(by_label[label])))
    print(f'\n--- {label.upper()} ---')
    for i, s in enumerate(samples, 1):
        print(f'  [{i}] {s[:120]}')